In [1]:
import pandas as pd
import seaborn as sns

sns.set_style("whitegrid")
sns.set_palette("Set2")

In [12]:
AUG_METHODS = ["none", "autoaugment", "standard", "standard_cutmix", "standard_mixup"]

data_seed_0 = pd.DataFrame()
data_seed_42 = pd.DataFrame()
data_seed_3407 = pd.DataFrame()

metrics_data = pd.DataFrame()

for aug in AUG_METHODS:
    aug_data_seed_0 = pd.read_json(f"../outputs/02_aug/mobilenet_{aug}/seed_0/epoch_metrics.json")
    aug_data_seed_0["augmentation"] = aug

    data_seed_0 = pd.concat([data_seed_0, aug_data_seed_0], ignore_index=True)

    val_seed_0 = (
        pd.read_json(
            f"../outputs/02_aug/mobilenet_{aug}/seed_0/metrics.json",
            typ="series",
        )
        .to_frame()
        .T
    )
    test_seed_0 = (
        pd.read_json(
            f"../outputs/02_aug/mobilenet_{aug}/seed_0/test_metrics.json",
            typ="series",
        )
        .to_frame()
        .T
    )

    test_seed_0 = pd.concat([val_seed_0, test_seed_0], ignore_index=True, axis=1)
    test_seed_0["seed"] = 0
    test_seed_0["augmentation"] = aug

    metrics_data = pd.concat([metrics_data, test_seed_0], ignore_index=True)

    aug_data_seed_42 = pd.read_json(f"../outputs/02_aug/mobilenet_{aug}/seed_42/epoch_metrics.json")
    aug_data_seed_42["augmentation"] = aug
    data_seed_42 = pd.concat([data_seed_42, aug_data_seed_42], ignore_index=True)

    val_seed_42 = (
        pd.read_json(
            f"../outputs/02_aug/mobilenet_{aug}/seed_42/metrics.json",
            typ="series",
        )
        .to_frame()
        .T
    )
    test_seed_42 = (
        pd.read_json(
            f"../outputs/02_aug/mobilenet_{aug}/seed_42/test_metrics.json",
            typ="series",
        )
        .to_frame()
        .T
    )
    test_seed_42 = pd.concat([val_seed_42, test_seed_42], ignore_index=True, axis=1)
    test_seed_42["seed"] = 42
    test_seed_42["augmentation"] = aug
    metrics_data = pd.concat([metrics_data, test_seed_42], ignore_index=True)

    aug_data_seed_3407 = pd.read_json(
        f"../outputs/02_aug/mobilenet_{aug}/seed_3407/epoch_metrics.json"
    )
    aug_data_seed_3407["augmentation"] = aug
    data_seed_3407 = pd.concat([data_seed_3407, aug_data_seed_3407], ignore_index=True)

    val_seed_3407 = (
        pd.read_json(
            f"../outputs/02_aug/mobilenet_{aug}/seed_3407/metrics.json",
            typ="series",
        )
        .to_frame()
        .T
    )
    test_seed_3407 = (
        pd.read_json(
            f"../outputs/02_aug/mobilenet_{aug}/seed_3407/test_metrics.json",
            typ="series",
        )
        .to_frame()
        .T
    )
    test_seed_3407 = pd.concat([val_seed_3407, test_seed_3407], ignore_index=True, axis=1)
    test_seed_3407["seed"] = 3407
    test_seed_3407["augmentation"] = aug
    metrics_data = pd.concat([metrics_data, test_seed_3407], ignore_index=True)

metrics_data.columns = [
    "best_val_accuracy",
    "best_val_loss",
    "completed_epochs",
    "resumed_from_epoch",
    "test_accuracy",
    "test_loss",
    "seed",
    "augmentation",
]

In [13]:
data_seed_0

,epoch,train_accuracy,train_loss,val_accuracy,val_loss,augmentation
0,1,0.295222,1.887280,0.355500,1.836934,none
1,2,0.390244,1.650238,0.409222,1.601251,none
2,3,0.426489,1.559814,0.425211,1.557153,none
3,4,0.446611,1.501358,0.442656,1.514480,none
4,5,0.465633,1.452323,0.451600,1.497766,none
...,...,...,...,...,...,...
295,56,0.488611,1.681941,0.544711,1.337904,standard_mixup
296,57,0.483822,1.688466,0.543400,1.346645,standard_mixup
297,58,0.490478,1.677645,0.542133,1.348376,standard_mixup
298,59,0.494256,1.672193,0.543389,1.345382,standard_mixup


In [16]:
metrics_data.groupby("augmentation").agg(
    {
        "test_accuracy": ["mean", "std"],
        "test_loss": ["mean", "std"],
        "best_val_accuracy": ["mean", "std"],
        "best_val_loss": ["mean", "std"],
    }
).sort_values(("test_accuracy", "mean"), ascending=False)

test_accuracy           test_loss           best_val_accuracy  \
                         mean       std      mean       std              mean   
augmentation                                                                    
standard             0.557856  0.005937  1.258245  0.011660          0.559656   
autoaugment          0.546937  0.005231  1.272264  0.010885          0.550144   
standard_mixup       0.541115  0.009038  1.347397  0.021192          0.543785   
standard_cutmix      0.540374  0.002340  1.345929  0.009889          0.542104   
none                 0.474907  0.008494  1.495643  0.020343          0.475244   

                          best_val_loss            
                      std          mean       std  
augmentation                                       
standard         0.004321      1.249866  0.011012  
autoaugment      0.005190      1.262815  0.012002  
standard_mixup   0.009594      1.341443  0.022412  
standard_cutmix  0.003434      1.340747  0.009673  
none             0.008111      1.494114  0.018791